# 🇵🇰 Pakistan NIH Surveillance Data

This notebook demonstrates how to access disease surveillance data from the **Pakistan National Institute of Health (NIH)** Integrated Disease Surveillance and Response (IDSR) system.

**Data Sources:**
- **IDSR Weekly Bulletins** - Weekly disease surveillance reports (priority diseases)
- **Provincial Reports** - Province and district-level surveillance data
- **Seasonal Awareness and Alert Letters (SAAL)** - Seasonal disease alerts
- **Advisories** - Public health advisories

**Coverage:**
- **Geographic**: All 4 provinces + Islamabad Capital Territory + Azad Kashmir + Gilgit-Baltistan
- **Diseases**: 30+ priority diseases including Dengue, Malaria, Typhoid, Hepatitis, COVID-19
- **Frequency**: Weekly bulletins, annual reports

**Requirements:**
```bash
pip install pandas matplotlib seaborn requests beautifulsoup4 pypdf
```

## 1. Setup and Imports

In [ ]:
import sys
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import Pakistan NIH accessor
from epidatasets.sources.pakistan_nih import PakistanNIHAccessor

print("✅ Imports completed successfully!")
print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2. Initialize Pakistan NIH Accessor

In [ ]:
# Initialize the Pakistan NIH accessor
nih = PakistanNIHAccessor()

print("🇵🇰 Pakistan NIH Accessor initialized")
print(f"Source: {nih.source_name}")
print(f"Portal: {nih.source_url}")
print(f"Base URL: {nih.BASE_URL}")

## 3. Explore Provinces and Territories

In [ ]:
# List Pakistani provinces and territories
provinces = nih.PROVINCES
print(f"\n🇵🇰 Pakistani Provinces and Territories ({len(provinces)} total):")
for i, prov in enumerate(provinces, 1):
    print(f"  {i}. {prov}")

# Visualize provinces
fig, ax = plt.subplots(figsize=(8, 6))
y_pos = range(len(provinces))
ax.barh(y_pos, [1]*len(provinces), color='darkgreen')
ax.set_yticks(y_pos)
ax.set_yticklabels(provinces, fontsize=10)
ax.set_xlabel('Count')
ax.set_title('Pakistani Provinces and Territories')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Priority Diseases Under Surveillance

In [ ]:
# List priority diseases under IDSR surveillance
diseases = nih.PRIORITY_DISEASES
print(f"\n🦠 Priority Diseases Under Surveillance ({len(diseases)} total):")
for i, disease in enumerate(diseases, 1):
    print(f"  {i}. {disease}")

# Visualize disease categories
fig, ax = plt.subplots(figsize=(8, 8))
y_pos = range(len(diseases))
colors = plt.cm.Reds([0.3 + 0.7 * i / len(diseases) for i in range(len(diseases))])
ax.barh(y_pos, [1]*len(diseases), color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels(diseases, fontsize=8)
ax.set_xlabel('Count')
ax.set_title('Pakistan NIH Priority Diseases Under IDSR Surveillance')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. List Available Weekly Bulletins

In [ ]:
# List available IDSR weekly bulletins for a year range
try:
    bulletins = nih.list_available_bulletins(start_year=2024, end_year=2025)
    print(f"\n📊 Available Weekly Bulletins ({len(bulletins)} found):")
    print(bulletins.head(10).to_string(index=False))

    # Plot bulletins by year
    if not bulletins.empty and 'year' in bulletins.columns:
        yearly_counts = bulletins['year'].value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(8, 4))
        yearly_counts.plot(kind='bar', ax=ax, color='steelblue')
        ax.set_xlabel('Year')
        ax.set_ylabel('Number of Bulletins')
        ax.set_title('IDSR Weekly Bulletins by Year')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"⚠️ Could not list bulletins: {e}")
    print("This may require network access to the NIH portal.")

## 6. Download and Extract a Weekly Bulletin

In [ ]:
# Download a specific weekly bulletin
try:
    bulletin = nih.get_weekly_bulletin(year=2024, week=51)
    print(f"\n📄 Bulletin downloaded:")
    print(f"  Year: {bulletin['year']}")
    print(f"  Week: {bulletin['week']}")
    print(f"  PDF Path: {bulletin['pdf_path']}")
    print(f"  URL: {bulletin['url']}")

    # Extract text from PDF
    text = nih._extract_pdf_text(bulletin['pdf_path'])
    print(f"\n📝 Extracted Text (first 1000 chars):")
    print(text[:1000] if text else "No text extracted")

    # Try to parse surveillance table
    if text:
        df = nih.parse_surveillance_table(text)
        if not df.empty:
            print(f"\n📊 Surveillance Data Extracted ({len(df)} rows):")
            print(df.head().to_string(index=False))
        else:
            print("\n⚠️ No structured surveillance table found in text.")
except Exception as e:
    print(f"⚠️ Could not download bulletin: {e}")
    print("This may require network access or the specific bulletin may not be available.")

## 7. Get Latest Bulletin

In [ ]:
# Fetch the most recent bulletin
try:
    latest = nih.get_latest_bulletin()
    if latest:
        print(f"\n📰 Latest Bulletin:")
        print(f"  Year: {latest['year']}")
        print(f"  Week: {latest['week']}")
        print(f"  PDF Path: {latest['pdf_path']}")
        print(f"  URL: {latest['url']}")
    else:
        print("\n⚠️ No bulletins found.")
except Exception as e:
    print(f"⚠️ Could not fetch latest bulletin: {e}")

## 8. Seasonal Awareness and Alert Letters (SAAL)

In [ ]:
# List Seasonal Awareness and Alert Letters
try:
    saals = nih.get_saals()
    print(f"\n🚨 Seasonal Awareness and Alert Letters ({len(saals)} found):")
    if not saals.empty:
        print(saals.head(10).to_string(index=False))
    else:
        print("  No SAALs available or could not fetch.")
except Exception as e:
    print(f"⚠️ Could not fetch SAALs: {e}")

## 9. Public Health Advisories

In [ ]:
# List public health advisories
try:
    advisories = nih.get_advisories()
    print(f"\n📢 Public Health Advisories ({len(advisories)} found):")
    if not advisories.empty:
        print(advisories.head(10).to_string(index=False))
    else:
        print("  No advisories available or could not fetch.")
except Exception as e:
    print(f"⚠️ Could not fetch advisories: {e}")

## 10. Provincial Reports

In [ ]:
# List provincial reports
try:
    provincial = nih.get_provincial_reports()
    print(f"\n🏛️ Provincial Reports ({len(provincial)} found):")
    if not provincial.empty:
        print(provincial.head(10).to_string(index=False))

        # Filter by specific province
        punjab_reports = nih.get_provincial_reports(province="Punjab")
        print(f"\n📍 Punjab-specific Reports ({len(punjab_reports)} found):")
        print(punjab_reports.head(5).to_string(index=False))
    else:
        print("  No provincial reports available or could not fetch.")
except Exception as e:
    print(f"⚠️ Could not fetch provincial reports: {e}")

## 11. Summary and Next Steps

This notebook demonstrated how to access Pakistan NIH surveillance data through the `PakistanNIHAccessor`.

**Key capabilities:**
- ✅ Access IDSR weekly bulletins with priority disease surveillance data
- ✅ Extract text and tables from PDF reports
- ✅ Browse provincial and district-level reports
- ✅ Access seasonal alerts and public health advisories

**Potential use cases:**
- Dengue outbreak monitoring across Pakistani provinces
- Seasonal disease trend analysis
- Cross-provincial comparison of surveillance indicators
- Integration with climate data for vector-borne disease modeling

**Documentation:**
- **NIH Portal**: https://phb.nih.org.pk/
- **IDSR Weekly**: https://phb.nih.org.pk/integratedisease-surveillance-and-response
- **Repository**: https://github.com/fccoelho/epidemiological-datasets